In [1]:
import pandas as pd

customer_clustering = pd.read_csv('data/customer_clustering.csv')
customer_clustering.head(), customer_clustering.shape

(       월평균값  월중앙값  월최댓값  월최솟값  회원기간  cluster
 0  4.833333   5.0     8     2    47        2
 1  5.083333   5.0     7     3    47        2
 2  4.583333   5.0     6     3    47        2
 3  4.833333   4.5     7     2    47        2
 4  3.916667   4.0     6     1    47        2,
 (4192, 6))

In [2]:
customer_join = pd.read_csv('data/customer_join.csv')
customer_join.shape
customer_join.head()


,customer_id,name,class,gender,start_date,end_date,campaign_id,is_deleted,class_name,price,...,max_x,min_x,routine_flg_x,mean_y,median_y,max_y,min_y,routine_flg_y,calc_date,membership_period
0,OA832399,XXXX,C01,F,2015-05-01,NaN,CA1,0,0_종일,10500,...,8,2,1,4.833333,5.0,8,2,1,2019-04-30,47
1,PL270116,XXXXX,C01,M,2015-05-01,NaN,CA1,0,0_종일,10500,...,7,3,1,5.083333,5.0,7,3,1,2019-04-30,47
2,OA974876,XXXXX,C01,M,2015-05-01,NaN,CA1,0,0_종일,10500,...,6,3,1,4.583333,5.0,6,3,1,2019-04-30,47
3,HD024127,XXXXX,C01,F,2015-05-01,NaN,CA1,0,0_종일,10500,...,7,2,1,4.833333,4.5,7,2,1,2019-04-30,47
4,HD661448,XXXXX,C03,F,2015-05-01,NaN,CA1,0,2_야간,6000,...,6,1,1,3.916667,4.0,6,1,1,2019-04-30,47


In [3]:
customer_clustering = pd.concat([customer_clustering, customer_join], axis=1)
customer_clustering.head(2)

,월평균값,월중앙값,월최댓값,월최솟값,회원기간,cluster,customer_id,name,class,gender,...,max_x,min_x,routine_flg_x,mean_y,median_y,max_y,min_y,routine_flg_y,calc_date,membership_period
0,4.833333,5.0,8,2,47,2,OA832399,XXXX,C01,F,...,8,2,1,4.833333,5.0,8,2,1,2019-04-30,47
1,5.083333,5.0,7,3,47,2,PL270116,XXXXX,C01,M,...,7,3,1,5.083333,5.0,7,3,1,2019-04-30,47


In [4]:
# 탈퇴 기준 
customer_clustering.groupby(['cluster','is_deleted'], as_index=False).count()[['cluster','is_deleted','customer_id']]
# customer_clustering.groupby(
#     ['cluster', 'is_deleted'],
#     as_index=False
# )['customer_id'].count().rename(columns={'customer_id': 'count'})


,cluster,is_deleted,customer_id
0,0,0,791
1,0,1,543
2,1,1,771
3,2,0,1231
4,2,1,18
5,3,0,820
6,3,1,18


-> 0,1번 그룹 관리 필요

In [5]:
# routine 기준 
customer_clustering.groupby(['cluster','routine_flg_x'], as_index=False).count()[['cluster','routine_flg_x','customer_id']]

,cluster,routine_flg_x,customer_id
0,0,0,227
1,0,1,1107
2,1,0,499
3,1,1,272
4,2,0,2
5,2,1,1247
6,3,0,51
7,3,1,787


In [6]:
summary = customer_clustering.groupby('cluster').agg({
    'customer_id':'count',
    'is_deleted':'mean',
    'routine_flg_x':'mean',
    '회원기간':'mean',
    '월평균값':'mean'
}).round(2)
summary

,customer_id,is_deleted,routine_flg_x,회원기간,월평균값
cluster,,,,,
0,1334,0.41,0.83,14.86,5.54
1,771,1.00,0.35,9.28,3.07
2,1249,0.01,1.00,36.92,4.68
3,838,0.02,0.94,7.02,8.06


In [7]:
summary.columns = ['인원','탈퇴율','정기비율','회원기간(개월)','월평균이용']
summary

,인원,탈퇴율,정기비율,회원기간(개월),월평균이용
cluster,,,,,
0,1334,0.41,0.83,14.86,5.54
1,771,1.00,0.35,9.28,3.07
2,1249,0.01,1.00,36.92,4.68
3,838,0.02,0.94,7.02,8.06


In [8]:
summary['페르소나'] = [
    '혼합 그룹(탈퇴 41%)',
    '전원 탈퇴',
    '장기 안정 단골(탈퇴 1%)',
    '신규의욕+지속(탈퇴 2%)'
]

summary

,인원,탈퇴율,정기비율,회원기간(개월),월평균이용,페르소나
cluster,,,,,,
0,1334,0.41,0.83,14.86,5.54,혼합 그룹(탈퇴 41%)
1,771,1.00,0.35,9.28,3.07,전원 탈퇴
2,1249,0.01,1.00,36.92,4.68,장기 안정 단골(탈퇴 1%)
3,838,0.02,0.94,7.02,8.06,신규의욕+지속(탈퇴 2%)


In [9]:
customer_clustering.to_csv('./data/customer_clustering2.csv',index=False)


### '이번달까지의 이용 패턴으로 다음 달 이용 횟수 예측 가능?'
#### 지도학습 -> 회귀(예측), 분류 

- 과거의 데이터 (시간의 흐름에 따른 이용 횟수 패턴)
- 2018년 11월: 종속변수(y)
- 2018년 5월 ~ 2018년 10월 : 독립변수(X)


In [10]:
uselog = pd.read_csv('data/use_log.csv')
uselog.shape

(197428, 3)

In [11]:
uselog.info()

<class 'pandas.DataFrame'>
RangeIndex: 197428 entries, 0 to 197427
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   log_id       197428 non-null  str  
 1   customer_id  197428 non-null  str  
 2   usedate      197428 non-null  str  
dtypes: str(3)
memory usage: 4.5 MB


In [12]:
uselog['usedate'] = pd.to_datetime(uselog['usedate'])
uselog.info()

<class 'pandas.DataFrame'>
RangeIndex: 197428 entries, 0 to 197427
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   log_id       197428 non-null  str           
 1   customer_id  197428 non-null  str           
 2   usedate      197428 non-null  datetime64[us]
dtypes: datetime64[us](1), str(2)
memory usage: 4.5 MB


In [13]:
uselog['연월'] = uselog['usedate'].dt.strftime('%Y%m')
uselog.head()

,log_id,customer_id,usedate,연월
0,L00000049012330,AS009373,2018-04-01,201804
1,L00000049012331,AS015315,2018-04-01,201804
2,L00000049012332,AS040841,2018-04-01,201804
3,L00000049012333,AS046594,2018-04-01,201804
4,L00000049012334,AS073285,2018-04-01,201804


In [14]:
uselog_months = uselog.groupby(['연월','customer_id'],as_index=False).count()
uselog_months.head()

,연월,customer_id,log_id,usedate
0,201804,AS002855,4,4
1,201804,AS009013,2,2
2,201804,AS009373,3,3
3,201804,AS015315,6,6
4,201804,AS015739,7,7


In [15]:
uselog_months.rename(columns={'log_id':'count'},inplace=True)
uselog_months.head()

,연월,customer_id,count,usedate
0,201804,AS002855,4,4
1,201804,AS009013,2,2
2,201804,AS009373,3,3
3,201804,AS015315,6,6
4,201804,AS015739,7,7


In [16]:
del uselog_months['usedate']
uselog_months.head()

,연월,customer_id,count
0,201804,AS002855,4
1,201804,AS009013,2
2,201804,AS009373,3
3,201804,AS015315,6
4,201804,AS015739,7


In [17]:
year_months = list(uselog_months['연월'].unique())
year_months

['201804',
 '201805',
 '201806',
 '201807',
 '201808',
 '201809',
 '201810',
 '201811',
 '201812',
 '201901',
 '201902',
 '201903']

In [18]:
predict_data = pd.DataFrame()

In [19]:
for i in range(6, len(year_months)):
    tmp = uselog_months.loc[uselog_months["연월"] == year_months[i]]
    tmp.rename(columns={"count":"count_pred"}, inplace=True)
    # print(i, year_months[i], tmp.columns)
    for j in range(1, 7):
        tmp_before = uselog_months.loc[uselog_months["연월"] == year_months[i-j]]
        tmp_before.rename(columns={"count":"count_{}".format(j-1)}, inplace=True)
        del tmp_before["연월"]
        tmp = pd.merge(tmp, tmp_before, how="left", on="customer_id")
        # print(year_months[i-j], tmp_before.columns)
    predict_data = pd.concat([predict_data, tmp], ignore_index=True)

In [20]:
predict_data.shape

(18310, 9)

In [21]:
predict_data.head()

,연월,customer_id,count_pred,count_0,count_1,count_2,count_3,count_4,count_5
0,201810,AS002855,3,7.0,3.0,5.0,5.0,5.0,4.0
1,201810,AS008805,2,2.0,5.0,7.0,8.0,NaN,NaN
2,201810,AS009373,5,6.0,6.0,7.0,4.0,4.0,3.0
3,201810,AS015233,7,9.0,11.0,5.0,7.0,7.0,NaN
4,201810,AS015315,4,7.0,3.0,6.0,3.0,3.0,6.0


In [22]:
predict_data = predict_data.dropna()
predict_data = predict_data.reset_index(drop=True)
predict_data.head()

,연월,customer_id,count_pred,count_0,count_1,count_2,count_3,count_4,count_5
0,201810,AS002855,3,7.0,3.0,5.0,5.0,5.0,4.0
1,201810,AS009373,5,6.0,6.0,7.0,4.0,4.0,3.0
2,201810,AS015315,4,7.0,3.0,6.0,3.0,3.0,6.0
3,201810,AS015739,5,6.0,5.0,8.0,6.0,5.0,7.0
4,201810,AS019860,7,5.0,7.0,4.0,6.0,8.0,6.0


In [23]:
predict_data.shape

(15113, 9)

In [24]:
predict_data = pd.merge(predict_data, customer_join[["customer_id", "start_date"]], how="left", on="customer_id")
predict_data.head(2)

,연월,customer_id,count_pred,count_0,count_1,count_2,count_3,count_4,count_5,start_date
0,201810,AS002855,3,7.0,3.0,5.0,5.0,5.0,4.0,2016-11-01
1,201810,AS009373,5,6.0,6.0,7.0,4.0,4.0,3.0,2015-11-01


In [25]:
predict_data["now_date"] = pd.to_datetime(predict_data["연월"], format="%Y%m")
predict_data["start_date"] = pd.to_datetime(predict_data["start_date"])

from dateutil.relativedelta import relativedelta

predict_data["period"] = None
for i in range(len(predict_data)):
    delta = relativedelta(predict_data["now_date"][i], predict_data["start_date"][i])
    # predict_data["period"][i] = delta.years * 12 + delta.months
    predict_data.loc[predict_data.index[i], "period"] = delta.years * 12 + delta.months

In [26]:
predict_data.head()

,연월,customer_id,count_pred,count_0,count_1,count_2,count_3,count_4,count_5,start_date,now_date,period
0,201810,AS002855,3,7.0,3.0,5.0,5.0,5.0,4.0,2016-11-01,2018-10-01,23
1,201810,AS009373,5,6.0,6.0,7.0,4.0,4.0,3.0,2015-11-01,2018-10-01,35
2,201810,AS015315,4,7.0,3.0,6.0,3.0,3.0,6.0,2015-07-01,2018-10-01,39
3,201810,AS015739,5,6.0,5.0,8.0,6.0,5.0,7.0,2017-06-01,2018-10-01,16
4,201810,AS019860,7,5.0,7.0,4.0,6.0,8.0,6.0,2017-10-01,2018-10-01,12


In [27]:
predict_data.columns

Index(['연월', 'customer_id', 'count_pred', 'count_0', 'count_1', 'count_2',
       'count_3', 'count_4', 'count_5', 'start_date', 'now_date', 'period'],
      dtype='str')

In [28]:
predict_data[
    [
        "count_pred",
        "count_0",
        "count_1",
        "count_2",
        "count_3",
        "count_4",
        "count_5",
        "period",
    ]
].describe().round(2)

,count_pred,count_0,count_1,count_2,count_3,count_4,count_5
count,15113.00,15113.00,15113.00,15113.00,15113.00,15113.00,15113.00
mean,4.97,5.08,5.18,5.30,5.44,5.61,5.79
std,1.89,1.89,1.91,1.93,1.94,1.95,2.01
min,1.00,1.00,1.00,1.00,1.00,1.00,1.00
25%,4.00,4.00,4.00,4.00,4.00,4.00,4.00
50%,5.00,5.00,5.00,5.00,5.00,6.00,6.00
75%,6.00,6.00,6.00,7.00,7.00,7.00,7.00
max,12.00,12.00,12.00,12.00,14.00,14.00,14.00


In [29]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [30]:
X = predict_data[["count_0", "count_1", "count_2", "count_3", "count_4", "count_5"]]
y = predict_data["count_pred"]
X.shape, y.shape

((15113, 6), (15113,))

In [31]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1)
X_train.shape, X_test.shape

((11334, 6), (3779, 6))

In [32]:
model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [33]:
# 결정계수 (R2) : 설명력. : 0 ~ 1 (1에 가까울 수록 좋은 성능)
print("학습 성능평가", model.score(X_train, y_train))
print("테스트 성능평가", model.score(X_test, y_test))

학습 성능평가 0.20532252000779905
테스트 성능평가 0.18221985599874546


In [34]:
predict_data["period"] = predict_data["period"].astype(int)
predict_data["period"].dtype

dtype('int64')

In [35]:
X = predict_data[["count_0", "count_1", "count_2", "count_3", "count_4", "count_5", "period"]]
y = predict_data["count_pred"]
X.shape, y.shape

((15113, 7), (15113,))

In [36]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1)
# X_train.shape, X_test.shape
model = LinearRegression()
model.fit(X_train, y_train)
print("학습 성능평가", model.score(X_train, y_train))
print("테스트 성능평가", model.score(X_test, y_test))

학습 성능평가 0.20603712620706338
테스트 성능평가 0.18359513033842212


In [37]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

In [38]:
predict_data = pd.DataFrame()
for i in range(4, len(year_months)):
    tmp = uselog_months.loc[uselog_months["연월"] == year_months[i]]
    tmp.rename(columns={"count": "count_pred"}, inplace=True)
    # print(i, year_months[i], tmp.columns)
    for j in range(1, 5):
        tmp_before = uselog_months.loc[uselog_months["연월"] == year_months[i - j]]
        tmp_before.rename(columns={"count": "count_{}".format(j - 1)}, inplace=True)
        del tmp_before["연월"]
        tmp = pd.merge(tmp, tmp_before, how="left", on="customer_id")
        # print(year_months[i-j], tmp_before.columns)
    predict_data = pd.concat([predict_data, tmp], ignore_index=True)

In [39]:
# y = ax + b
# a : 가중치, 기울기
model.coef_

# b : 절편, 상수항
model.intercept_

np.float64(1.1924178064232702)

In [40]:
X_test.iloc[0]

count_0     7.0
count_1     4.0
count_2     2.0
count_3     2.0
count_4     1.0
count_5     4.0
period     41.0
Name: 4406, dtype: float64

In [41]:
# [0.16231052, 0.15174496, 0.14699171, 0.122303, 0.08428464, 0.0194099, 0.00570869]
y = ((0.162*7) + (0.151*4) + (0.146*2) + (0.122*2) + (0.084*1) + (0.019*4) + (0.005*41)) + 1.192
y

3.8310000000000004

In [42]:
# 두명의 회원 정보를 가지고 예측
x1 = [3, 4, 4, 6, 8, 7, 8]
x2 = [2, 2, 3, 3, 4, 6, 8]
x_pred = [x1, x2]
x_pred

model.predict(x_pred)

c:\Users\UserK\work\05_pandas_basic\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([4.46392996, 3.12768036])

In [43]:
uselog_months.to_csv("./data/uselog_months.csv", index=False)

"스포츠 센터에는 ... 탈퇴하려면 월말까지 신청하면 그 다음 달 말에 탈퇴가 됩니다."

시점	사건
8월 말 (8/31)	회원이 탈퇴 신청
9월 한 달	회원이 여전히 사용 가능 (탈퇴 진행 중)
9월 말 (9/30)	실제 탈퇴 완료 → end_date = 2018-09-30
탈퇴 예측의 목적은 "탈퇴를 미연에 방지". 그러므로 우리가 예측해야 할 시점은 **9월(end_date)이 아니라 8월(신청 시점)**입니다. 9월 데이터로 예측해 봐야 이미 신청한 후라 늦었어요.

"왜 end_date 컬럼의 탈퇴 월이 아닌 탈퇴 전월의 데이터를 작성할까요? 탈퇴를 예측하는 목적은 탈퇴를 미연에 방지하는 것입니다. 이 스포츠 센터에서는 월말까지 탈퇴 신청을 해야 다음 달 말에 탈퇴할 수 있습니다. 예를 들어, 2018년 9월 30일에 탈퇴한 회원이 있다고 합시다. 이 경우, 8월에 탈퇴 신청을 했기 때문에 9월의 데이터를 사용한다면 탈퇴를 미연에 방지할 수 없습니다. 그래서 탈퇴 월을 2018년 8월로 하고, 그 1개월 전인 7월의 데이터로부터 8월에 탈퇴 신청을 할 확률을 예측해야 합니다."

그러면 어떻게 예측? 7월 데이터로 8월 탈퇴 신청을 예측하는 모델을 만듭니다. 즉:

목적(종속) 변수: 그 회원이 그달에 탈퇴 신청을 했는가 (0/1)
설명(독립) 변수: 그달과 1개월 전의 이용 횟수 + 정적 정보 (캠페인·클래스·성별·정기성·기간)



In [44]:
exit_customer = customer_join.loc[customer_join["is_deleted"] ==  1]
exit_customer.head(2)

,customer_id,name,class,gender,start_date,end_date,campaign_id,is_deleted,class_name,price,...,max_x,min_x,routine_flg_x,mean_y,median_y,max_y,min_y,routine_flg_y,calc_date,membership_period
708,TS511179,XXXXXX,C01,F,2016-05-01,2018-04-30,CA1,1,0_종일,10500,...,3,3,0,3.0,3.0,3,3,0,2018-04-30,23
729,TS443736,XXXX,C02,M,2016-05-01,2018-04-30,CA1,1,1_주간,7500,...,3,3,0,3.0,3.0,3,3,0,2018-04-30,23


In [45]:
exit_customer["exit_date"] = None

In [46]:
exit_customer["end_date"] = pd.to_datetime(exit_customer["end_date"])

In [47]:
for i in range(len(exit_customer)):
  exit_customer.loc[exit_customer.index[i], "exit_date"] = exit_customer["end_date"].iloc[i] - relativedelta(months=1)

exit_customer.head(2)

,customer_id,name,class,gender,start_date,end_date,campaign_id,is_deleted,class_name,price,...,min_x,routine_flg_x,mean_y,median_y,max_y,min_y,routine_flg_y,calc_date,membership_period,exit_date
708,TS511179,XXXXXX,C01,F,2016-05-01,2018-04-30,CA1,1,0_종일,10500,...,3,0,3.0,3.0,3,3,0,2018-04-30,23,2018-03-30 00:00:00
729,TS443736,XXXX,C02,M,2016-05-01,2018-04-30,CA1,1,1_주간,7500,...,3,0,3.0,3.0,3,3,0,2018-04-30,23,2018-03-30 00:00:00


In [48]:
exit_customer["exit_date"] = pd.to_datetime(exit_customer["exit_date"])

exit_customer["연월"] = exit_customer["exit_date"].dt.strftime("%Y%m")
exit_customer.head(2)

,customer_id,name,class,gender,start_date,end_date,campaign_id,is_deleted,class_name,price,...,routine_flg_x,mean_y,median_y,max_y,min_y,routine_flg_y,calc_date,membership_period,exit_date,연월
708,TS511179,XXXXXX,C01,F,2016-05-01,2018-04-30,CA1,1,0_종일,10500,...,0,3.0,3.0,3,3,0,2018-04-30,23,2018-03-30,201803
729,TS443736,XXXX,C02,M,2016-05-01,2018-04-30,CA1,1,1_주간,7500,...,0,3.0,3.0,3,3,0,2018-04-30,23,2018-03-30,201803


In [49]:
uselog_months.head(2)

,연월,customer_id,count
0,201804,AS002855,4
1,201804,AS009013,2


In [50]:
customer_join.head(2)

,customer_id,name,class,gender,start_date,end_date,campaign_id,is_deleted,class_name,price,...,max_x,min_x,routine_flg_x,mean_y,median_y,max_y,min_y,routine_flg_y,calc_date,membership_period
0,OA832399,XXXX,C01,F,2015-05-01,NaN,CA1,0,0_종일,10500,...,8,2,1,4.833333,5.0,8,2,1,2019-04-30,47
1,PL270116,XXXXX,C01,M,2015-05-01,NaN,CA1,0,0_종일,10500,...,7,3,1,5.083333,5.0,7,3,1,2019-04-30,47


In [51]:
exit_customer = customer_join.loc[customer_join["is_deleted"] ==  1]
exit_customer.head(2)

exit_customer["exit_date"] = None

exit_customer["end_date"] = pd.to_datetime(exit_customer["end_date"])

for i in range(len(exit_customer)):
  exit_customer.loc[exit_customer.index[i], "exit_date"] = exit_customer["end_date"].iloc[i] - relativedelta(months=1)

exit_customer.head(2)

exit_customer["exit_date"] = pd.to_datetime(exit_customer["exit_date"])

exit_customer["연월"] = exit_customer["exit_date"].dt.strftime("%Y%m")
exit_customer.head(2)

exit_customer.head(2)

uselog_months.head(2)

year_months = None
uselog = None
tmp = None
tmp_before = None

# 이전 2둘의 이용 횟수 데이터 생성
year_months = list(uselog_months["연월"].unique())
uselog = pd.DataFrame()
for i in range(1, len(year_months)):
  # print(i)
  tmp = uselog_months.loc[uselog_months["연월"] == year_months[i]]
  # print(tmp.columns)
  tmp.rename(columns={"count":"count_0"}, inplace=True)
  tmp_before = uselog_months.loc[uselog_months["연월"] == year_months[i-1]]
  tmp_before.rename(columns={"count":"count_1"}, inplace=True)
  del tmp_before["연월"]
  tmp = pd.merge(tmp, tmp_before, how="left", on="customer_id")
  uselog = pd.concat([uselog, tmp], ignore_index=True)

uselog.isnull().sum()

uselog.head()

uselog.info()

exit_uselog = pd.merge(uselog, exit_customer, how="left", on=["customer_id", "연월"])
exit_uselog.shape

exit_uselog.head()

exit_uselog = exit_uselog.dropna(subset=["name"])

exit_uselog.shape

len(exit_uselog["customer_id"].unique())

len(exit_uselog["is_deleted"].unique())

# 언더 샘플링, 오버 샘플링
conti_customer = customer_join.loc[customer_join["is_deleted"] == 0]
conti_uselog = pd.merge(uselog, conti_customer, how="left", on=["customer_id"])
conti_uselog.shape

conti_uselog.head(2)

conti_uselog = conti_uselog.dropna(subset=["name"])
conti_uselog.shape

<class 'pandas.DataFrame'>
RangeIndex: 33851 entries, 0 to 33850
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   연월           33851 non-null  str    
 1   customer_id  33851 non-null  str    
 2   count_0      33851 non-null  int64  
 3   count_1      32650 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 1.0 MB


(27422, 26)

In [52]:
# SMOTE
# 언더샘플링

conti_uselog = conti_uselog.sample(frac=1).reset_index(drop=True)
conti_uselog = conti_uselog.drop_duplicates(subset='customer_id')


In [53]:
predict_data = pd.concat([conti_uselog, exit_uselog], ignore_index=True)
predict_data.shape

(3946, 27)

In [54]:
predict_data.head()

,연월,customer_id,count_0,count_1,name,class,gender,start_date,end_date,campaign_id,...,min_x,routine_flg_x,mean_y,median_y,max_y,min_y,routine_flg_y,calc_date,membership_period,exit_date
0,201808,OA758034,4,2.0,XXX,C01,F,2016-09-01,NaN,CA1,...,2.0,1.0,4.916667,4.5,9.0,2.0,1.0,2019-04-30,31.0,NaT
1,201902,PL507166,4,3.0,XXX,C01,M,2015-09-01,NaN,CA1,...,3.0,1.0,4.083333,4.0,7.0,3.0,1.0,2019-04-30,43.0,NaT
2,201808,TS953790,10,9.0,XXXXX,C02,F,2018-01-01,NaN,CA1,...,6.0,1.0,8.166667,8.0,11.0,6.0,1.0,2019-04-30,15.0,NaT
3,201902,AS929514,6,6.0,XXXXXX,C01,M,2017-04-01,NaN,CA1,...,3.0,1.0,5.500000,5.0,9.0,3.0,1.0,2019-04-30,24.0,NaT
4,201901,GD648354,2,4.0,XXXXX,C01,M,2016-11-01,NaN,CA1,...,2.0,1.0,5.083333,5.0,8.0,2.0,1.0,2019-04-30,29.0,NaT
